In [ ]:
import os, warnings,sys,time    

CUDA_VISIBLE_DEVICES=""
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0,'../../uqmodels/abench')
sys.path.insert(0,'../../n5_uqmodels/')
import abench
import uqmodels
import yaml

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Specification of Data :
import yaml
config_benchmark_path = "config/config_benchmark.yaml"
with open(config_benchmark_path) as f:
    config_benchmark = yaml.safe_load(f)
    
for key,value in config_benchmark.items():
    print(key,value)

# Metric Specification 
from src.metric import base_rmse,ABMetricGeneric
#['Id'0, 'timestamp'1, 'positionX'2, 'positionY'3, 'positionZ'4, 'sizeX'5,'sizeY+', 'sizeZ7', 'VelX'8, 'VelY'9, 'VelZ'10, 'Vel'11, 'Class'12, 'rot_x'13,'rot_y'14, 'edge'15, 'ts'16, 'departure'17, 'destination18, 'dataset'19, 'set'20,'filename'21, 'seqId'22, 'length'23, 'cat_length'24, 'cat_edge'25],
dict_sets_configs_grid1={'context_mask':[0],'context_dim_mask':1,'context_variable_ids':[[24]]}
metrics = [ABMetricGeneric(base_rmse, name="RMSE",reduce=True),
           ABMetricGeneric(base_rmse, name="RMSE_grid",dict_sets_config=dict_sets_configs_grid1,reduce=True)]

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# Spécification of DataExperiment Plan
from src.data_loader import get_DataExperiment
DataExperiment = get_DataExperiment(config_benchmark)

# Spécification of Models Candidates
from src.component import build_params,get_model_constructor

dict_comp = {}
for name,config_path in config_benchmark['Components_config'].items():
    dict_comp[name]={'module':get_model_constructor(name),'parameters':build_params(config_path)}

# Specification of the Component candidate list :
exp_design=[]
for key,model_builder in dict_comp.items():
    subexp_design=[{'name':key,'model':key}]
    exp_design.append(subexp_design)

from src.component import ComponentAE
dict_exp={'Component': ComponentAE,
          'tuning_scheme' : {},
          'model': dict_comp,
          'exp_design':exp_design}

# Run benchmark with train step

In [ ]:
from src.metric import base_rmse,ABMetricGeneric
AB_RMSE = ABMetricGeneric(metric=base_rmse,name="rmse", mask=None, dim_mask=None, list_ctx_constraint=None,reduce=True)
list_metrics = [AB_RMSE]

In [ ]:
# Metric Specification 
from abench.benchmark.benchmark import benchmark
storing = 'Results'
benchmark(storing=storing,
          ABDataExperiment=DataExperiment,
          dict_exp=dict_exp,
          # Component_class,
          list_metrics=list_metrics,verbose=True)

# Run only inference for a new validation set

In [ ]:
# Metric Specification 
from src.metric import base_rmse,ABMetricGeneric
AB_RMSE = ABMetricGeneric(metric=base_rmse,name="rmse", mask=None, dim_mask=None, list_ctx_constraint=None,reduce=True)
list_metrics = [AB_RMSE]
from abench.benchmark import benchmark

storing = 'Results/'
from src.data_loader import get_DataExperiment
config_benchmark['validation_config'] = {'Noisy_eps': {'constraint_selection': [['dataset',['dataset1_noisy_eps']]], 'constraint_rejection': []}}
DataExperiment = get_DataExperiment(config_benchmark,with_test=False)
DataExperiment.name = 'cv_experiment_additional'

# List of component name
list_component_name = list(config_benchmark['Components_config'].keys())

# Dict making association between name and class
from src.component import ComponentAE
Component_class_dict = {}
for name_model in list_component_name:
    Component_class_dict[name_model] = ComponentAE

benchmark.inference(storing=storing,
                    ABDataExperiment=DataExperiment,
                    list_component_name = list_component_name,
                    Component_class_dict = Component_class_dict,
                    list_metrics=list_metrics,verbose=True)

In [ ]:
# Update full data experiment 
import yaml
config_benchmark_path = "config/config_benchmark.yaml"
with open(config_benchmark_path) as f:
    config_benchmark = yaml.safe_load(f)

storing = 'Results/'
from src.data_loader import get_DataExperiment
config_benchmark['validation_config']['Noisy_eps'] = {'constraint_selection': [['dataset',['dataset1_noisy_eps']]], 'constraint_rejection': []}
DataExperiment = get_DataExperiment(config_benchmark,with_test=True)
from abench.store.api import store_ABDataExperiment
store_ABDataExperiment(storing,DataExperiment)